# 43 – ACS Demographics by TOC / TOD

This notebook summarizes **basic ACS demographics** for each TOC / TOD.

**Goals:**
- Load tract+ACS layer and TOC/TOD polygons.
- Ensure CRS is consistent.
- Intersect tracts with TOCs.
- Compute tract area shares within each TOC.
- Create simple TOC-level demographics (e.g., population-weighted).


In [230]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CENSUS_DIR = DATA_DIR / "census" / "processed"
BOUNDARIES_DIR = DATA_DIR / "boundaries"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

tracts_acs_path = CENSUS_DIR / "tracts_acs_2019_2023.gpkg"
toc_clean_path = BOUNDARIES_DIR / "processed" / "toc_clean.gpkg"

tracts_acs_path, toc_clean_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census/processed/tracts_acs_2019_2023.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/processed/toc_clean.gpkg'))

In [231]:
tracts_acs = gpd.read_file(tracts_acs_path)
toc = gpd.read_file(toc_clean_path)

print("Tracts CRS:", tracts_acs.crs)
print("TOC CRS:", toc.crs)


Tracts CRS: EPSG:2868
TOC CRS: EPSG:2868


In [232]:
# anxietying CRS differences

PROJECT_CRS = "EPSG:2868"

if toc.crs != PROJECT_CRS:
    toc = toc.to_crs(PROJECT_CRS)
if tracts_acs.crs != PROJECT_CRS:
    tracts_acs = tracts_acs.to_crs(PROJECT_CRS)

tracts_acs.crs, toc.crs

(<Projected CRS: EPSG:2868>
 Name: NAD83(HARN) / Arizona Central (ft)
 Axis Info [cartesian]:
 - X[east]: Easting (foot)
 - Y[north]: Northing (foot)
 Area of Use:
 - name: United States (USA) - Arizona - counties Coconino; Maricopa; Pima; Pinal; Santa Cruz; Yavapai.
 - bounds: (-113.35, 31.33, -110.44, 37.01)
 Coordinate Operation:
 - name: SPCS83 Arizona Central zone (international foot)
 - method: Transverse Mercator
 Datum: NAD83 (High Accuracy Reference Network)
 - Ellipsoid: GRS 1980
 - Prime Meridian: Greenwich,
 <Projected CRS: EPSG:2868>
 Name: NAD83(HARN) / Arizona Central (ft)
 Axis Info [cartesian]:
 - X[east]: Easting (foot)
 - Y[north]: Northing (foot)
 Area of Use:
 - name: United States (USA) - Arizona - counties Coconino; Maricopa; Pima; Pinal; Santa Cruz; Yavapai.
 - bounds: (-113.35, 31.33, -110.44, 37.01)
 Coordinate Operation:
 - name: SPCS83 Arizona Central zone (international foot)
 - method: Transverse Mercator
 Datum: NAD83 (High Accuracy Reference Network)
 - 

In [233]:
tracts_acs.columns

Index(['GISJOIN', 'ASNQE001', 'ASQPE001', 'ASVBE001', 'ASS8E001', 'ASS8E002',
       'ASS8E003', 'ASS9E001', 'ASS9E002', 'ASS9E003', 'ASORE001', 'ASORE003',
       'ASORE004', 'ASORE010', 'ASORE018', 'ASORE019', 'ASORE021', 'ASORE002',
       'geometry'],
      dtype='object')

In [234]:
toc.columns

Index(['OBJECTID', 'APPLICABIL', 'geometry'], dtype='object')

In [235]:
TRACT_GEOID_FIELD = "GISJOIN"
TOTAL_POPULATION = "ASNQE001"         # Total population (Sex by Age: total)
MEDIAN_HOUSEHOLD_INCOME = "ASQPE001"     # Median household income (2023 inflation-adjusted $)
MEDIAN_RENT = "ASVBE001"       # Median gross rent (dollars)
HOUSING_TOTAL = "ASS8E001"  # Total housing units
OCCUPIED_HOUSEHOLDS = "ASS8E002"     # Occupied housing units (households)
VACANT_UNITS = "ASS8E003"     # Vacant housing units
TOTAL_OCCUPIED_UNITS = "ASS9E001"  # Total occupied housing units (same universe as ASS8E002)
OWNER_OCCUPIED = "ASS9E002"     # Owner-occupied
RENTER_OCCUPIED = "ASS9E003"     # Renter-occupied
TOTAL_WORKERS_16P = "ASORE001"  # Total workers 16+ (denominator)
DRIVE_ALONE = "ASORE003"     # Drove alone
CARPOOLED = "ASORE004"     # Carpooled
PUBLIC_TRANSPORT = "ASORE010"     # Public transportation (excluding taxicab, all modes)
BICYCLE = "ASORE018"     # Bicycle
WALKED = "ASORE019"     # Walked
WORKED_FROM_HOME = "ASORE021"     # Worked from home
AUTO_COMMUTERS = "ASORE002"     # Car, truck, or van (all auto commuters)


tracts_acs[[TRACT_GEOID_FIELD, TOTAL_POPULATION, MEDIAN_HOUSEHOLD_INCOME, MEDIAN_RENT, HOUSING_TOTAL, OCCUPIED_HOUSEHOLDS, VACANT_UNITS, TOTAL_OCCUPIED_UNITS, OWNER_OCCUPIED, RENTER_OCCUPIED, TOTAL_WORKERS_16P, DRIVE_ALONE, CARPOOLED, PUBLIC_TRANSPORT, BICYCLE, WALKED, WORKED_FROM_HOME, AUTO_COMMUTERS]].head()

,GISJOIN,ASNQE001,ASQPE001,ASVBE001,ASS8E001,ASS8E002,ASS8E003,ASS9E001,ASS9E002,ASS9E003,ASORE001,ASORE003,ASORE004,ASORE010,ASORE018,ASORE019,ASORE021,ASORE002
0,G0400130010102,6265,188486,-666666666,3537,2824,713,2824,2731,93,2016,931,93,22,19,110,773,1024
1,G0400130010103,3681,117813,2744,1884,1516,368,1516,1451,65,1735,1188,60,0,0,0,462,1248
2,G0400130010104,3131,140587,-666666666,2447,1737,710,1737,1681,56,915,316,0,0,0,0,584,316
3,G0400130030401,4744,145865,1679,3170,2423,747,2423,2221,202,1563,596,152,0,0,59,734,748
4,G0400130030402,4153,108031,2096,2273,1928,345,1928,1803,125,1717,960,37,0,0,32,642,997


In [236]:
toc.head()

,OBJECTID,APPLICABIL,geometry
0,1,TOD District - Gateway,"POLYGON ((663431.678 894322.427, 663434.469 89..."
1,2,TOD District - Eastlake Garfield,"POLYGON ((654751.939 894390.665, 654764.736 89..."
2,3,TOD District - Midtown,"POLYGON ((654767.886 907514.625, 654767.631 90..."
3,4,TOD District - Uptown,"POLYGON ((649448.684 915530.44, 654791.762 915..."
4,5,TOD District - Solano,"POLYGON ((646830.036 919545.608, 646829.436 91..."


In [237]:
tracts_acs[[
    TRACT_GEOID_FIELD,
    TOTAL_POPULATION,
    MEDIAN_HOUSEHOLD_INCOME,
    MEDIAN_RENT
]].dtypes

GISJOIN     object
ASNQE001    object
ASQPE001    object
ASVBE001    object
dtype: object

In [238]:
# since they're objects, convert to numeric

acs_cols = [
    TOTAL_POPULATION, MEDIAN_HOUSEHOLD_INCOME, MEDIAN_RENT,
    HOUSING_TOTAL, OCCUPIED_HOUSEHOLDS, VACANT_UNITS,
    OWNER_OCCUPIED, RENTER_OCCUPIED,
    TOTAL_WORKERS_16P, DRIVE_ALONE, CARPOOLED, PUBLIC_TRANSPORT,
    BICYCLE, WALKED, WORKED_FROM_HOME, AUTO_COMMUTERS
]

for col in acs_cols:
    tracts_acs[col] = pd.to_numeric(tracts_acs[col], errors="coerce")

In [239]:
# time to calculate a shitload of metrics!

# population per acre

tracts_acs["pop_per_acre"] = (
    tracts_acs["geometry"].area / 43560
).rdiv(tracts_acs[TOTAL_POPULATION])

# households per acre

tracts_acs["hh_per_acre"] = tracts_acs[OCCUPIED_HOUSEHOLDS] / (tracts_acs["geometry"].area / 43560)

# housing vacancy rate

tracts_acs["vacancy_rate"] = tracts_acs[VACANT_UNITS] / tracts_acs[HOUSING_TOTAL]

# owner vs renter share

tracts_acs["pct_owner"] = tracts_acs[OWNER_OCCUPIED] / tracts_acs[OCCUPIED_HOUSEHOLDS]
tracts_acs["pct_renter"] = tracts_acs[RENTER_OCCUPIED] / tracts_acs[OCCUPIED_HOUSEHOLDS]

# commute mode shares

tracts_acs["pct_drive_alone"] = tracts_acs[DRIVE_ALONE] / tracts_acs[TOTAL_WORKERS_16P]
tracts_acs["pct_carpool"]     = tracts_acs[CARPOOLED] / tracts_acs[TOTAL_WORKERS_16P]
tracts_acs["pct_transit"]     = tracts_acs[PUBLIC_TRANSPORT] / tracts_acs[TOTAL_WORKERS_16P]
tracts_acs["pct_bike"]        = tracts_acs[BICYCLE] / tracts_acs[TOTAL_WORKERS_16P]
tracts_acs["pct_walk"]        = tracts_acs[WALKED] / tracts_acs[TOTAL_WORKERS_16P]
tracts_acs["pct_wfh"]         = tracts_acs[WORKED_FROM_HOME] / tracts_acs[TOTAL_WORKERS_16P]

In [240]:
# formatting labels:

# tracts_acs["income_label"] = tracts_acs[MEDIAN_HOUSEHOLD_INCOME].map(
 #    lambda v: f"${v:,.0f}" if pd.notnull(v) else "N/A"
# )

# tracts_acs["rent_label"] = tracts_acs[MEDIAN_RENT].map(
#    lambda v: f"${v:,.0f}" if pd.notnull(v) else "N/A"
# )

# pct_columns = [
#    "vacancy_rate", "pct_owner", "pct_renter",
#    "pct_drive_alone", "pct_carpool", "pct_transit",
#    "pct_bike", "pct_walk", "pct_wfh"
# ]

# for col in pct_columns:
#     tracts_acs[col + "_label"] = tracts_acs[col].map(
#         lambda v: f"{v*100:,.1f}%" if pd.notnull(v) else "N/A"
#     )

In [241]:
tracts_acs.head()

,GISJOIN,ASNQE001,ASQPE001,ASVBE001,ASS8E001,ASS8E002,ASS8E003,ASS9E001,ASS9E002,ASS9E003,...,hh_per_acre,vacancy_rate,pct_owner,pct_renter,pct_drive_alone,pct_carpool,pct_transit,pct_bike,pct_walk,pct_wfh
0,G0400130010102,6265,188486,-666666666,3537,2824,713,2824,2731,93,...,0.004172,0.201583,0.967068,0.032932,0.461806,0.046131,0.010913,0.009425,0.054563,0.383433
1,G0400130010103,3681,117813,2744,1884,1516,368,1516,1451,65,...,0.162941,0.195329,0.957124,0.042876,0.684726,0.034582,0.000000,0.000000,0.000000,0.266282
2,G0400130010104,3131,140587,-666666666,2447,1737,710,1737,1681,56,...,0.427313,0.290151,0.967761,0.032239,0.345355,0.000000,0.000000,0.000000,0.000000,0.638251
3,G0400130030401,4744,145865,1679,3170,2423,747,2423,2221,202,...,0.369416,0.235647,0.916632,0.083368,0.381318,0.097249,0.000000,0.000000,0.037748,0.469610
4,G0400130030402,4153,108031,2096,2273,1928,345,1928,1803,125,...,0.133840,0.151782,0.935166,0.064834,0.559115,0.021549,0.000000,0.000000,0.018637,0.373908


In [242]:
# hallelujah, we now have metrics! time to begin spatial intersection with TOCs

import geopandas as gpd
import pandas as pd
import numpy as np

# Make a working copy of tracts with ACS
tracts_dem = tracts_acs.copy()

# Ensure CRS is projected (2868) so area is in square feet
if tracts_dem.crs is None:
    tracts_dem = tracts_dem.set_crs(2868)
else:
    tracts_dem = tracts_dem.to_crs(2868)

# Same for TOCs
tocs_dem = toc.copy()
if tocs_dem.crs is None:
    tocs_dem = tocs_dem.set_crs(2868)
else:
    tocs_dem = tocs_dem.to_crs(2868)

# Give TOCs a stable ID if they don't already have one
if "TOC_ID" not in tocs_dem.columns:
    tocs_dem = tocs_dem.reset_index().rename(columns={"index": "TOC_ID"})

# Precompute tract area in square feet so it survives overlay
tracts_dem["tract_area_sqft"] = tracts_dem.geometry.area


In [243]:
tocs_dem.head()

,TOC_ID,OBJECTID,APPLICABIL,geometry
0,0,1,TOD District - Gateway,"POLYGON ((663431.678 894322.427, 663434.469 89..."
1,1,2,TOD District - Eastlake Garfield,"POLYGON ((654751.939 894390.665, 654764.736 89..."
2,2,3,TOD District - Midtown,"POLYGON ((654767.886 907514.625, 654767.631 90..."
3,3,4,TOD District - Uptown,"POLYGON ((649448.684 915530.44, 654791.762 915..."
4,4,5,TOD District - Solano,"POLYGON ((646830.036 919545.608, 646829.436 91..."


In [244]:
tracts_x_toc = gpd.overlay(
    tracts_dem,
    tocs_dem,
    how="intersection",
    keep_geom_type=False
)

print("Rows in tract × TOC overlay:", len(tracts_x_toc))

Rows in tract × TOC overlay: 130


In [245]:
# Intersection area
tracts_x_toc["intersect_sqft"] = tracts_x_toc.geometry.area
tracts_x_toc["intersect_acres"] = tracts_x_toc["intersect_sqft"] / 43560.0

# Area weight = share of each tract's area that falls inside this TOC
tracts_x_toc["area_weight"] = (
    tracts_x_toc["intersect_sqft"] /
    tracts_x_toc["tract_area_sqft"]
)

# tract total area
tracts_x_toc["tract_sqft"] = tracts_x_toc.groupby("GISJOIN")["intersect_sqft"].transform("sum")

tracts_x_toc["area_weight"].describe()

count    130.000000
mean       0.432775
std        0.419226
min        0.000002
25%        0.001504
50%        0.336048
75%        0.990226
max        1.000000
Name: area_weight, dtype: float64

In [246]:
COUNT_FIELDS = [
    TOTAL_POPULATION,
    HOUSING_TOTAL,
    OCCUPIED_HOUSEHOLDS,
    VACANT_UNITS,
    OWNER_OCCUPIED,
    RENTER_OCCUPIED,
    TOTAL_WORKERS_16P,
    DRIVE_ALONE,
    CARPOOLED,
    PUBLIC_TRANSPORT,
    BICYCLE,
    WALKED,
    WORKED_FROM_HOME,
    AUTO_COMMUTERS,
]

for col in COUNT_FIELDS:
    tracts_x_toc[col + "_w"] = tracts_x_toc[col] * tracts_x_toc["area_weight"]

In [247]:
group_cols = ["TOC_ID", "APPLICABIL"]

# Build aggregation dict: sum all the *_contrib fields + acres
agg_dict = {
    "intersect_acres": "sum"
}
for col in COUNT_FIELDS:
    agg_dict[col + "_w"] = "sum"

toc_dem = (
    tracts_x_toc
    .groupby(group_cols, as_index=False)
    .agg(agg_dict)
)

# Clean column names: strip "_w"
rename_dict = {col + "_w": col for col in COUNT_FIELDS}
toc_dem = toc_dem.rename(columns=rename_dict)

toc_dem.head()


,TOC_ID,APPLICABIL,intersect_acres,ASNQE001,ASS8E001,ASS8E002,ASS8E003,ASS9E002,ASS9E003,ASORE001,ASORE003,ASORE004,ASORE010,ASORE018,ASORE019,ASORE021,ASORE002
0,0,TOD District - Gateway,2418.944362,13919.344291,4802.854853,4404.074570,398.780283,1195.807028,3208.267542,6462.513573,4068.090989,1129.578085,444.958189,80.237735,109.625509,449.088289,5197.669073
1,1,TOD District - Eastlake Garfield,1220.462979,7826.045040,4021.409613,3379.818960,641.590653,914.607318,2465.211643,3878.897524,2429.796737,450.745541,93.812812,24.819494,169.119522,612.777992,2880.542278
2,2,TOD District - Midtown,1283.000217,11830.041147,8259.602020,7079.314543,1180.287477,2359.838005,4719.476538,7267.383607,4042.675520,695.478293,502.473774,18.990820,309.705123,1576.935853,4738.153812
3,3,TOD District - Uptown,1332.064285,10889.582934,5910.686353,5546.764459,363.921894,2049.764274,3497.000185,6514.910696,4319.818878,165.468790,266.655946,39.419910,112.935643,1513.787255,4485.287669
4,4,TOD District - Solano,1100.265433,14440.725434,6067.435580,5518.487681,548.947899,1864.403058,3654.084623,7270.143922,4213.907706,816.450440,398.735435,24.968847,359.417134,1223.597030,5030.358146


In [248]:
toc_dem["pct_drive_alone"] = 100 * toc_dem[DRIVE_ALONE] / toc_dem[TOTAL_WORKERS_16P]
toc_dem["pct_transit"]     = 100 * toc_dem[PUBLIC_TRANSPORT] / toc_dem[TOTAL_WORKERS_16P]
toc_dem["pct_bike"]        = 100 * toc_dem[BICYCLE] / toc_dem[TOTAL_WORKERS_16P]
toc_dem["pct_walk"]        = 100 * toc_dem[WALKED] / toc_dem[TOTAL_WORKERS_16P]
toc_dem["pct_wfh"]         = 100 * toc_dem[WORKED_FROM_HOME] / toc_dem[TOTAL_WORKERS_16P]
toc_dem["pct_auto"]        = 100 * toc_dem[AUTO_COMMUTERS] / toc_dem[TOTAL_WORKERS_16P]

toc_dem.head()

,TOC_ID,APPLICABIL,intersect_acres,ASNQE001,ASS8E001,ASS8E002,ASS8E003,ASS9E002,ASS9E003,ASORE001,...,ASORE018,ASORE019,ASORE021,ASORE002,pct_drive_alone,pct_transit,pct_bike,pct_walk,pct_wfh,pct_auto
0,0,TOD District - Gateway,2418.944362,13919.344291,4802.854853,4404.074570,398.780283,1195.807028,3208.267542,6462.513573,...,80.237735,109.625509,449.088289,5197.669073,62.949051,6.885219,1.241587,1.696329,6.949127,80.427979
1,1,TOD District - Eastlake Garfield,1220.462979,7826.045040,4021.409613,3379.818960,641.590653,914.607318,2465.211643,3878.897524,...,24.819494,169.119522,612.777992,2880.542278,62.641426,2.418543,0.639859,4.359989,15.797736,74.261881
2,2,TOD District - Midtown,1283.000217,11830.041147,8259.602020,7079.314543,1180.287477,2359.838005,4719.476538,7267.383607,...,18.990820,309.705123,1576.935853,4738.153812,55.627661,6.914095,0.261316,4.261577,21.698811,65.197519
3,3,TOD District - Uptown,1332.064285,10889.582934,5910.686353,5546.764459,363.921894,2049.764274,3497.000185,6514.910696,...,39.419910,112.935643,1513.787255,4485.287669,66.306648,4.093010,0.605072,1.733495,23.235733,68.846495
4,4,TOD District - Solano,1100.265433,14440.725434,6067.435580,5518.487681,548.947899,1864.403058,3654.084623,7270.143922,...,24.968847,359.417134,1223.597030,5030.358146,57.961820,5.484560,0.343444,4.943742,16.830438,69.192002


In [249]:
# attach to TOC geometry

tocs_with_dem = tocs_dem.merge(
    toc_dem,
    on=["TOC_ID", "APPLICABIL"],
    how="left"
)

# true TOC area based on TOC polygons
tocs_with_dem["toc_area_sqft"] = tocs_with_dem.geometry.area
tocs_with_dem["toc_acres"] = tocs_with_dem["toc_area_sqft"] / 43560.0

# Population per acre
tocs_with_dem["pop_per_acre"] = (
    tocs_with_dem[TOTAL_POPULATION] /
    tocs_with_dem["toc_acres"]
)

# Households per acre
tocs_with_dem["hh_per_acre"] = (
    tocs_with_dem[OCCUPIED_HOUSEHOLDS] /
    tocs_with_dem["toc_acres"]
)

In [250]:
# income and rent approximations: tract median * households * area_weight
tracts_x_toc["income_weighted"] = (
    tracts_x_toc[MEDIAN_HOUSEHOLD_INCOME].fillna(0) *
    tracts_x_toc[OCCUPIED_HOUSEHOLDS].fillna(0) *
    tracts_x_toc["area_weight"]
)

tracts_x_toc["rent_weighted"] = (
    tracts_x_toc[MEDIAN_RENT].fillna(0) *
    tracts_x_toc[OCCUPIED_HOUSEHOLDS].fillna(0) *
    tracts_x_toc["area_weight"]
)

toc_income = (
    tracts_x_toc
    .groupby(["TOC_ID", "APPLICABIL"], as_index=False)
    .agg({
        "income_weighted": "sum",
        "rent_weighted": "sum",
        OCCUPIED_HOUSEHOLDS + "_w": "sum"
    })
)

toc_income = toc_income.rename(columns={
    OCCUPIED_HOUSEHOLDS + "_w": "hh_total"
})

toc_income["median_income_approx"] = toc_income["income_weighted"] / toc_income["hh_total"]
toc_income["median_rent_approx"]   = toc_income["rent_weighted"] / toc_income["hh_total"]

In [251]:
# merge everything back into tocs_with_dem

tocs_with_dem = toc_income.merge(
    toc_dem,
    on=["TOC_ID", "APPLICABIL"],
    how="left"
)

tocs_with_dem.head(11)

,TOC_ID,APPLICABIL,income_weighted,rent_weighted,hh_total,median_income_approx,median_rent_approx,intersect_acres,ASNQE001,ASS8E001,...,ASORE018,ASORE019,ASORE021,ASORE002,pct_drive_alone,pct_transit,pct_bike,pct_walk,pct_wfh,pct_auto
0,0,TOD District - Gateway,2.099283e+08,5.097215e+06,4404.074570,47666.824758,1157.386086,2418.944362,13919.344291,4802.854853,...,80.237735,109.625509,449.088289,5197.669073,62.949051,6.885219,1.241587,1.696329,6.949127,80.427979
1,1,TOD District - Eastlake Garfield,1.613483e+08,3.709420e+06,3379.818960,47738.728035,1097.520452,1220.462979,7826.045040,4021.409613,...,24.819494,169.119522,612.777992,2880.542278,62.641426,2.418543,0.639859,4.359989,15.797736,74.261881
2,2,TOD District - Midtown,5.228343e+08,1.150085e+07,7079.314543,73853.809662,1624.571152,1283.000217,11830.041147,8259.602020,...,18.990820,309.705123,1576.935853,4738.153812,55.627661,6.914095,0.261316,4.261577,21.698811,65.197519
3,3,TOD District - Uptown,4.255024e+08,8.094779e+06,5546.764459,76711.815672,1459.369509,1332.064285,10889.582934,5910.686353,...,39.419910,112.935643,1513.787255,4485.287669,66.306648,4.093010,0.605072,1.733495,23.235733,68.846495
4,4,TOD District - Solano,2.998474e+08,7.064961e+06,5518.487681,54335.073430,1280.234957,1100.265433,14440.725434,6067.435580,...,24.968847,359.417134,1223.597030,5030.358146,57.961820,5.484560,0.343444,4.943742,16.830438,69.192002
5,5,TOD District - 19North,6.341370e+08,1.334089e+07,10430.136344,60798.535299,1279.071912,1780.947386,22399.496066,11195.738128,...,112.236139,187.747876,1774.814622,8323.108262,60.620135,5.314936,0.999783,1.672430,15.809781,74.140993
6,6,50th Street Station Area,7.584102e+07,1.843089e+06,1397.124710,54283.640432,1319.201558,617.105659,3048.586703,1521.297287,...,13.014820,23.771741,365.670654,1382.606314,63.878418,4.357335,0.690553,1.261305,19.402120,73.359713
7,7,Capitol Extension Area,1.196863e+08,2.520710e+06,2063.514028,58001.224814,1221.561930,1116.466203,5339.173857,2372.854914,...,48.209766,0.230218,363.948503,2134.264228,64.798414,4.853094,1.792639,0.008560,13.533112,79.360778
8,8,I-10 West Extension Area,1.271326e+09,2.939138e+07,22828.810637,55689.542302,1287.468572,7735.922063,77864.054330,23805.602737,...,176.771654,311.258268,2221.514970,29945.651488,67.878337,3.040533,0.512905,0.903119,6.445746,86.887576
9,9,Northwest Extension Area II,3.291577e+08,7.308436e+06,5457.748156,60310.174629,1339.093742,1740.704483,13958.933685,5826.741859,...,20.440853,110.315789,581.080449,5212.851623,59.873332,4.929296,0.315743,1.704013,8.975765,80.521258


In [252]:
toc_dem_path = BOUNDARIES_DIR / "tocs" / "tocs_demographics_cleaned.csv"

tocs_with_dem.to_csv(toc_dem_path, index=False)
print("Saved TOC demographics:", toc_dem_path)

Saved TOC demographics: c:\Users\arjav\DevSource\toc-performance-dashboard\data\boundaries\tocs\tocs_demographics_cleaned.csv
